# 02 Preprocessing for Open Prediction

## 2.0 Purpose

This notebook prepares the frozen modelling dataset for open prediction. It starts from the saved recipient-campaign modelling dataframe and prepares train/test datasets and preprocessing rules. No model training is performed in this notebook.

## 2.1 Imports and file paths

In [78]:
# imports

import pandas as pd
import numpy as np

from pathlib import Path

# display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [79]:
# project paths
PROJECT_ROOT = Path.home() / "Documents" / "Thesis"

DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DATA_DIR = DATA_DIR / "Processed Data"
OUTPUT_DIR = PROJECT_ROOT / "Outputs"

OPEN_MODEL_PATH = PROCESSED_DATA_DIR / "df_open_model_v1.parquet"

print("Project root:", PROJECT_ROOT)
print("Processed data folder exists:", PROCESSED_DATA_DIR.exists())
print("Open model file exists:", OPEN_MODEL_PATH.exists())

Project root: /Users/khanhhien/Documents/Thesis
Processed data folder exists: True
Open model file exists: True


## 2.2 Load frozen modelling datasets

In [80]:
df_open = pd.read_parquet(OPEN_MODEL_PATH)

print("Shape:", df_open.shape)

display(df_open.head())

display(
    df_open["open"]
    .value_counts(normalize=True)
)

Shape: (1057544, 38)


,user_id,mailing_id,open,click,mailing_info,subject_line,preheader,send_date,send_day_of_week,date_trusted,has_campaign_metadata,gender,age_clean,age_group,n_interests,interest_topic_match,campaign_topic,subject_missing,subject_length,subject_length_group,subject_contains_number,subject_contains_promotion,subject_contains_exclamation,preheader_missing,preheader_length,preheader_length_group,preheader_contains_number,preheader_contains_promotion,preheader_contains_exclamation,prior_email_count,prior_open_count,prior_click_count,historical_open_rate,historical_click_rate,historical_open_bucket,prior_email_frequency_bucket,had_prior_click,is_first_email
0,5c6bebde5798a794283224c9,668,0.0,0.0,2025/04/24 CPX - MM MAX Magazine - 4 gratis nrs,"Lezen, puzzelen, genieten: 4 edities cadeau","Vraag nu aan – geen kosten, geen verplichtingen.",2025-04-25,Vrijdag,True,1,male,NaN,NaN,17,1,Media & Publishing,False,43,medium,True,True,False,False,48,medium,False,False,False,0,0.0,0.0,NaN,NaN,NaN,very_little,False,True
1,5c6bebde5798a794283224c9,691,0.0,0.0,2025/05/29 WOONCOMFORT AIRCO,Bespaar tot 80% op je energiekosten met een airco,Vraag nu gratis maatwerkadvies aan voor energi...,2025-04-30,Woensdag,True,1,male,NaN,NaN,17,0,Energy & Sustainability,False,49,long,True,False,False,False,63,long,False,True,False,1,0.0,0.0,0.0,0.0,very_low,very_little,False,False
2,5c6bebde5798a794283224c9,714,0.0,0.0,2025/05/06 CPX - MM - ENGIE BONUS,Profiteer nu: €300 bonus én vaste energietarieven,ENGIE helpt je graag met persoonlijk advies.,2025-05-07,Woensdag,True,1,male,NaN,NaN,17,0,Energy & Sustainability,False,49,long,True,True,False,False,44,short,False,False,False,2,0.0,0.0,0.0,0.0,very_low,very_little,False,False
3,5c6bebde5798a794283224c9,733,0.0,0.0,2025/05/14 LIFEPOINTS,Jouw mening is geld waard – start vandaag nog,Verdien punten voor digitale cadeaubonnen zoal...,2025-05-14,Woensdag,True,1,male,NaN,NaN,17,1,Lottery & Games,False,45,medium,False,False,False,False,65,long,False,True,False,3,0.0,0.0,0.0,0.0,very_low,very_little,False,False
4,5c6bebde5798a794283224c9,761,0.0,0.0,2025/05/27 CPX - MM - ENGIE,Speel mee met de ENGIE woordlegger!,"En win een solar powerbank t.w.v. €79,95.",2025-06-04,Woensdag,True,1,male,NaN,NaN,17,0,Energy & Sustainability,False,35,short,False,False,True,False,41,short,True,True,False,4,0.0,0.0,0.0,0.0,very_low,very_little,False,False


open
1.0    0.53618
0.0    0.46382
Name: proportion, dtype: float64

## 2.3 Select modelling target and dataset

- Target = open
- Dataset = df_open

## 2.4 Define column roles

In [81]:
# column audit
column_audit = pd.DataFrame({
    "column": df_open.columns,
    "dtype": df_open.dtypes.astype(str).values,
    "missing_rate": df_open.isna().mean().values
})

display(column_audit)

,column,dtype,missing_rate
0,user_id,object,0.000000
1,mailing_id,Int64,0.000000
2,open,float64,0.000000
3,click,float64,0.043825
4,mailing_info,object,0.025269
5,subject_line,object,0.025269
6,preheader,object,0.029937
7,send_date,datetime64[us],0.329770
8,send_day_of_week,object,0.329770
9,date_trusted,bool,0.000000


In [82]:
# define column roles
target_cols = [
    "open"
]

other_target_cols = [
    "click"
]

id_cols = [
    "user_id",
    "mailing_id"
]

audit_only_cols = [
    "mailing_info",
    "subject_line",
    "preheader",
    "send_date",
    "send_day_of_week",
    "date_trusted"
]

candidate_feature_cols = [
    "has_campaign_metadata",
    "gender",
    "age_clean",
    "age_group",
    "n_interests",
    "interest_topic_match",
    "campaign_topic",
    "subject_missing",
    "subject_length",
    "subject_length_group",
    "subject_contains_number",
    "subject_contains_promotion",
    "subject_contains_exclamation",
    "preheader_missing",
    "preheader_length",
    "preheader_length_group",
    "preheader_contains_number",
    "preheader_contains_promotion",
    "preheader_contains_exclamation",
    "prior_email_count",
    "prior_open_count",
    "prior_click_count",
    "historical_open_rate",
    "historical_click_rate",
    "historical_open_bucket",
    "prior_email_frequency_bucket",
    "had_prior_click",
    "is_first_email"
]

In [83]:
# assign column role
def assign_column_role(col):
    if col in target_cols:
        return "target"
    if col in other_target_cols:
        return "other_target_exclude"
    if col in id_cols:
        return "id_exclude"
    if col in audit_only_cols:
        return "audit_exclude"
    if col in candidate_feature_cols:
        return "candidate_feature"
    return "unclassified"


column_audit["role"] = column_audit["column"].apply(assign_column_role)

display(column_audit.sort_values(["role", "column"]))

,column,dtype,missing_rate,role
9,date_trusted,bool,0.000000,audit_exclude
4,mailing_info,object,0.025269,audit_exclude
6,preheader,object,0.029937,audit_exclude
7,send_date,datetime64[us],0.329770,audit_exclude
8,send_day_of_week,object,0.329770,audit_exclude
5,subject_line,object,0.025269,audit_exclude
12,age_clean,float64,0.745815,candidate_feature
13,age_group,category,0.745815,candidate_feature
16,campaign_topic,object,0.000000,candidate_feature
11,gender,object,0.000000,candidate_feature


In [84]:
# check
display(column_audit["role"].value_counts())

unclassified_cols = column_audit.loc[
    column_audit["role"] == "unclassified",
    "column"
].tolist()

print("Unclassified columns:", unclassified_cols)

role
candidate_feature       28
audit_exclude            6
id_exclude               2
target                   1
other_target_exclude     1
Name: count, dtype: int64

Unclassified columns: []


## 2.5 Define feature sets

In [85]:
# feature set A - core thesis model
feature_set_A_core = [
    # recipient features
    "gender",
    "age_group",
    "n_interests",
    "interest_topic_match",

    # campaign/content features
    "campaign_topic",
    "has_campaign_metadata",
    "subject_length_group",
    "subject_contains_number",
    "subject_contains_promotion",
    "preheader_length_group",
    "preheader_contains_number",
    "preheader_contains_promotion",

    # historical behaviour features
    "historical_open_bucket",
    "prior_email_frequency_bucket",
    "is_first_email"
]

In [86]:
# feature set B - core thesis model + interactions
interaction_source_cols = [
    ("campaign_topic", "age_group"),
    ("historical_open_bucket", "prior_email_frequency_bucket")
]

feature_set_B_interactions = feature_set_A_core.copy()

In [87]:
# feature set C - expanded feature set
feature_set_C_expanded = [
    # recipient features
    "gender",
    "age_group",
    "age_clean",
    "n_interests",
    "interest_topic_match",

    # campaign/content features
    "campaign_topic",
    "has_campaign_metadata",
    "subject_missing",
    "subject_length",
    "subject_length_group",
    "subject_contains_number",
    "subject_contains_promotion",
    "subject_contains_exclamation",
    "preheader_missing",
    "preheader_length",
    "preheader_length_group",
    "preheader_contains_number",
    "preheader_contains_promotion",
    "preheader_contains_exclamation",

    # historical behaviour features
    "prior_email_count",
    "prior_open_count",
    "prior_click_count",
    "historical_open_rate",
    "historical_click_rate",
    "historical_open_bucket",
    "prior_email_frequency_bucket",
    "had_prior_click",
    "is_first_email"
]

In [88]:
# validate
feature_sets = {
    "A_core": feature_set_A_core,
    "B_interactions": feature_set_B_interactions,
    "C_expanded": feature_set_C_expanded
}

for name, features in feature_sets.items():
    missing_features = [col for col in features if col not in df_open.columns]
    print(name)
    print("Number of features:", len(features))
    print("Missing features:", missing_features)
    print()

A_core
Number of features: 15
Missing features: []

B_interactions
Number of features: 15
Missing features: []

C_expanded
Number of features: 28
Missing features: []



## 2.6 Define evaluation and validation strategy

The modelling workflow uses a two-level evaluation structure.

First, the latest 20% of campaigns, ordered by `mailing_id`, are reserved as the final holdout test set. This test set represents future campaign prediction and must not be used for feature selection, model comparison, hyperparameter tuning, threshold tuning, or preprocessing decisions.

Second, the earliest 80% of campaigns form the train-validation pool. Model selection and validation will be performed only inside this pool using cross-validation.

Because each campaign is sent to many users, splitting must respect campaign groups. The same `mailing_id` should never appear in both training and validation/test data. Therefore, validation inside the train-validation pool will use grouped cross-validation with `mailing_id` as the group.

For open prediction, StratifiedGroupKFold will be used so that folds preserve the open-rate distribution as much as possible while keeping campaigns grouped. Preprocessing must be fitted only on the training part of each fold, not before cross-validation.

In [89]:
# evaluation strategy
TARGET_COL = "open"
GROUP_COL = "mailing_id"

FINAL_TEST_SIZE = 0.20
TRAINVAL_SIZE = 1 - FINAL_TEST_SIZE

INNER_CV_SPLITS = 5

print("Target column:", TARGET_COL)
print("Group column:", GROUP_COL)
print("Train-validation share:", TRAINVAL_SIZE)
print("Final test share:", FINAL_TEST_SIZE)
print("Inner CV folds:", INNER_CV_SPLITS)

Target column: open
Group column: mailing_id
Train-validation share: 0.8
Final test share: 0.2
Inner CV folds: 5


## 2.7 Create final chronological campaign holdout

The final holdout split is created at the campaign level. Campaigns are ordered by `mailing_id`, which is used as the available chronological proxy. The earliest 80% of campaigns form the train-validation pool, while the latest 20% are reserved as the final untouched test set.

In [90]:
# campaign-level chronological final holdout
campaign_order = sorted(df_open[GROUP_COL].unique())

n_campaigns = len(campaign_order)
test_start_idx = int(n_campaigns * TRAINVAL_SIZE)

trainval_campaigns = campaign_order[:test_start_idx]
test_campaigns = campaign_order[test_start_idx:]

trainval_df = df_open[df_open[GROUP_COL].isin(trainval_campaigns)].copy()
test_df = df_open[df_open[GROUP_COL].isin(test_campaigns)].copy()

print("Total campaigns:", n_campaigns)
print("Train-validation campaigns:", len(trainval_campaigns))
print("Final test campaigns:", len(test_campaigns))

print("\nTrain-validation shape:", trainval_df.shape)
print("Final test shape:", test_df.shape)

print("\nFirst trainval campaign:", min(trainval_campaigns))
print("Last trainval campaign:", max(trainval_campaigns))
print("First test campaign:", min(test_campaigns))
print("Last test campaign:", max(test_campaigns))

Total campaigns: 274
Train-validation campaigns: 219
Final test campaigns: 55

Train-validation shape: (781826, 38)
Final test shape: (275718, 38)

First trainval campaign: 653
Last trainval campaign: 1252
First test campaign: 1257
Last test campaign: 1418


In [91]:
# check campaign overlap
campaign_overlap = set(trainval_campaigns).intersection(set(test_campaigns))

print("Campaign overlap:", len(campaign_overlap))

assert len(campaign_overlap) == 0, "Campaign leakage: some mailing_ids appear in both trainval and test."
assert len(trainval_df) + len(test_df) == len(df_open), "Row counts do not add up."

Campaign overlap: 0


In [92]:
# check open distribution after splitting
print("Trainval open rate:")
display(
    trainval_df["open"]
    .value_counts(normalize=True)
)

print("Test open rate:")
display(
    test_df["open"]
    .value_counts(normalize=True)
)

Trainval open rate:


open
1.0    0.573273
0.0    0.426727
Name: proportion, dtype: float64

Test open rate:


open
0.0    0.569002
1.0    0.430998
Name: proportion, dtype: float64

The final test period has a lower open rate than the train-validation pool, indicating possible campaign-period distribution shift.

In [93]:
# check click distribution after splitting
print("Trainval click rate:")
display(
    trainval_df["click"]
    .value_counts(normalize=True, dropna=False)
)

print("Test click rate:")
display(
    test_df["click"]
    .value_counts(normalize=True, dropna=False)
)

Trainval click rate:


click
0.0    0.953968
NaN    0.044868
1.0    0.001164
Name: proportion, dtype: float64

Test click rate:


click
0.0    0.958512
NaN    0.040868
1.0    0.000620
Name: proportion, dtype: float64

Click is not needed for open prediction and should not be used as a feature. Checking it here is only an audit. The fact that test click rate is lower confirms later campaigns may differ, but it does not affect the open-preprocessing plan.

## 2.8 Check final holdout quality

This section checks whether the chronological final holdout split creates major distribution differences between the train-validation pool and the final test set. The test set remains untouched for modelling decisions, but its distribution is audited to document possible campaign-period shift.

In [94]:
# basic split summary
split_summary = pd.DataFrame({
    "split": ["trainval", "test"],
    "n_rows": [len(trainval_df), len(test_df)],
    "n_campaigns": [
        trainval_df[GROUP_COL].nunique(),
        test_df[GROUP_COL].nunique()
    ],
    "open_rate": [
        trainval_df[TARGET_COL].mean(),
        test_df[TARGET_COL].mean()
    ],
    "metadata_missing_rate": [
        1 - trainval_df["has_campaign_metadata"].mean(),
        1 - test_df["has_campaign_metadata"].mean()
    ]
})

display(split_summary)

,split,n_rows,n_campaigns,open_rate,metadata_missing_rate
0,trainval,781826,219,0.573273,0.03418
1,test,275718,55,0.430998,0.00000


In [95]:
# campaign topic distribution
topic_dist = pd.DataFrame({
    "trainval": trainval_df["campaign_topic"].value_counts(normalize=True),
    "test": test_df["campaign_topic"].value_counts(normalize=True)
}).fillna(0)

topic_dist["difference_test_minus_trainval"] = (
    topic_dist["test"] - topic_dist["trainval"]
)

display(topic_dist.sort_values("difference_test_minus_trainval", ascending=False))

,trainval,test,difference_test_minus_trainval
campaign_topic,,,
Media & Publishing,0.365780,0.539265,0.173485
Retail & Promotion,0.001569,0.078646,0.077076
Lottery & Games,0.100117,0.126263,0.026146
Food & Beverages,0.011274,0.018686,0.007412
Finance & Investment,0.012902,0.015073,0.002172
Telecom & Technology,0.001806,0.000000,-0.001806
Business,0.022171,0.007011,-0.015160
Automotive & Mobility,0.240653,0.215057,-0.025597
Health & Wellbeing,0.029842,0.000000,-0.029842


Test has much more:
- Media & Publishing
- Retail & Promotion

Test has none of:
- Charity & Social Impact
- Energy & Sustainability
- Health & Wellbeing
- Metadata Missing

In [96]:
# missingness comparison
missing_compare = pd.DataFrame({
    "trainval_missing": trainval_df.isna().mean(),
    "test_missing": test_df.isna().mean()
})

missing_compare["difference_test_minus_trainval"] = (
    missing_compare["test_missing"] - missing_compare["trainval_missing"]
)

display(
    missing_compare
    .sort_values("difference_test_minus_trainval", ascending=False)
    .head(15)
)

,trainval_missing,test_missing,difference_test_minus_trainval
send_date,0.324103,0.345839,0.021736
send_day_of_week,0.324103,0.345839,0.021736
age_group,0.743214,0.753190,0.009976
age_clean,0.743214,0.753190,0.009976
user_id,0.000000,0.000000,0.000000
preheader_length_group,0.000000,0.000000,0.000000
subject_contains_promotion,0.000000,0.000000,0.000000
subject_contains_exclamation,0.000000,0.000000,0.000000
preheader_missing,0.000000,0.000000,0.000000
preheader_length,0.000000,0.000000,0.000000


The chronological final holdout shows a lower open rate and a different campaign-topic composition than the train-validation pool. This suggests campaign-period distribution shift. The split is retained because it reflects the intended future-campaign prediction setting, but final test results should be interpreted as performance under temporal/topic shift.

## 2.9 Define inner CV strategy

Inner validation will be performed only inside the train-validation pool. The final test set remains untouched.

For open prediction, the inner validation strategy uses StratifiedGroupKFold. This keeps all rows from the same campaign together through `mailing_id`, while also trying to preserve the open-rate distribution across folds.

In [97]:
from sklearn.model_selection import StratifiedGroupKFold

inner_cv = StratifiedGroupKFold(
    n_splits=INNER_CV_SPLITS,
    shuffle=True,
    random_state=42
)

X_cv_reference = trainval_df[feature_set_A_core]
y_cv_reference = trainval_df[TARGET_COL]
groups_cv_reference = trainval_df[GROUP_COL]

print(inner_cv)
print("Reference X shape:", X_cv_reference.shape)
print("Reference y shape:", y_cv_reference.shape)
print("Reference groups:", groups_cv_reference.nunique())

StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)
Reference X shape: (781826, 15)
Reference y shape: (781826,)
Reference groups: 219


## 2.10 Check inner CV fold quality

This section checks whether the StratifiedGroupKFold split produces usable validation folds. Each fold should have no campaign overlap between training and validation, and the validation folds should have reasonably similar open rates.

In [98]:
fold_summaries = []

for fold_id, (train_idx, val_idx) in enumerate(
    inner_cv.split(X_cv_reference, y_cv_reference, groups=groups_cv_reference),
    start=1
):
    fold_train = trainval_df.iloc[train_idx]
    fold_val = trainval_df.iloc[val_idx]

    train_campaigns_fold = set(fold_train[GROUP_COL].unique())
    val_campaigns_fold = set(fold_val[GROUP_COL].unique())
    campaign_overlap = train_campaigns_fold.intersection(val_campaigns_fold)

    fold_summaries.append({
        "fold": fold_id,
        "train_rows": len(fold_train),
        "val_rows": len(fold_val),
        "train_campaigns": fold_train[GROUP_COL].nunique(),
        "val_campaigns": fold_val[GROUP_COL].nunique(),
        "campaign_overlap": len(campaign_overlap),
        "train_open_rate": fold_train[TARGET_COL].mean(),
        "val_open_rate": fold_val[TARGET_COL].mean(),
        "val_open_positive_count": int(fold_val[TARGET_COL].sum()),
        "val_open_negative_count": int((fold_val[TARGET_COL] == 0).sum())
    })

fold_quality = pd.DataFrame(fold_summaries)

display(fold_quality)

,fold,train_rows,val_rows,train_campaigns,val_campaigns,campaign_overlap,train_open_rate,val_open_rate,val_open_positive_count,val_open_negative_count
0,1,622701,159125,173,46,0,0.561046,0.621122,98836,60289
1,2,640613,141213,180,39,0,0.575329,0.563949,79637,61576
2,3,598719,183107,167,52,0,0.572579,0.575543,105386,77721
3,4,616035,165791,180,39,0,0.570283,0.584386,96886,68905
4,5,649236,132590,176,43,0,0.586451,0.508749,67455,65135


Overall trainval open rate is stable (~57.3%)

Validation open rates (50.9% - 62.1% -> 11% spread). But campaign-level grouping, campaigns have different performance, and only 219 campaigns -> perfect stratification is impossible

In [99]:
assert (fold_quality["campaign_overlap"] == 0).all(), "Campaign leakage found in CV folds."
assert (fold_quality["val_open_positive_count"] > 0).all(), "Some validation folds have no positive open cases."
assert (fold_quality["val_open_negative_count"] > 0).all(), "Some validation folds have no negative open cases."

## 2.11 Create optional interaction columns

Interaction columns are created deterministically by combining existing categorical variables. This does not learn from the data, so it does not create preprocessing leakage. The actual one-hot encoding of these interaction variables will still be fitted only inside the training fold during modelling.

In [100]:
# create interaction columns
def combine_interaction_cols(df, col1, col2):
    return (
        df[col1].astype("object").fillna("Missing").astype(str)
        + " x "
        + df[col2].astype("object").fillna("Missing").astype(str)
    )

for data in [trainval_df, test_df]:
    data["campaign_topic_x_age_group"] = combine_interaction_cols(
        data,
        "campaign_topic",
        "age_group"
    )

    data["historical_open_bucket_x_prior_email_frequency_bucket"] = combine_interaction_cols(
        data,
        "historical_open_bucket",
        "prior_email_frequency_bucket"
    )

In [101]:
# validate
interaction_cols = [
    "campaign_topic_x_age_group",
    "historical_open_bucket_x_prior_email_frequency_bucket"
]

for col in interaction_cols:
    print(col)
    print("Trainval unique values:", trainval_df[col].nunique())
    print("Test unique values:", test_df[col].nunique())
    print("Trainval missing:", trainval_df[col].isna().mean())
    print("Test missing:", test_df[col].isna().mean())
    print()

campaign_topic_x_age_group
Trainval unique values: 60
Test unique values: 35
Trainval missing: 0.0
Test missing: 0.0

historical_open_bucket_x_prior_email_frequency_bucket
Trainval unique values: 26
Test unique values: 26
Trainval missing: 0.0
Test missing: 0.0



## 2.12 Update feature sets with interaction columns


Feature Set B extends the core thesis feature set by adding two theoretically motivated interaction variables. Feature Set A remains the core main-effects specification, while Feature Set B represents the interaction specification.

In [102]:
# update feature set B with interaction columns
interaction_features = [
    "campaign_topic_x_age_group",
    "historical_open_bucket_x_prior_email_frequency_bucket"
]

feature_set_B_interactions = feature_set_A_core + interaction_features

feature_sets = {
    "A_core": feature_set_A_core,
    "B_interactions": feature_set_B_interactions,
    "C_expanded": feature_set_C_expanded
}

for name, features in feature_sets.items():
    missing_features = [
        col for col in features
        if col not in trainval_df.columns
    ]

    print(name)
    print("Number of features:", len(features))
    print("Missing features:", missing_features)
    print()

A_core
Number of features: 15
Missing features: []

B_interactions
Number of features: 17
Missing features: []

C_expanded
Number of features: 28
Missing features: []



## 2.13 Define feature types by feature set

Features are classified into numerical, categorical, and binary groups so that different preprocessing rules can be applied later. Numerical variables will be imputed and scaled, categorical variables will be imputed and one-hot encoded, and binary variables will be passed through after basic imputation if needed.

In [103]:
# feature type definitions for Feature Set A
feature_types_A = {
    "numeric": [
        "n_interests"
    ],

    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket"
    ],

    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_contains_number",
        "subject_contains_promotion",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "is_first_email"
    ]
}

In [104]:
# feature type definitions for Feature Set B = A + two interaction categorical features
feature_types_B = {
    "numeric": feature_types_A["numeric"].copy(),

    "categorical": (
        feature_types_A["categorical"].copy()
        + [
            "campaign_topic_x_age_group",
            "historical_open_bucket_x_prior_email_frequency_bucket"
        ]
    ),

    "binary": feature_types_A["binary"].copy()
}

In [105]:
# feature type definitions for Feature Set C
feature_types_C = {
    "numeric": [
        "age_clean",
        "n_interests",
        "subject_length",
        "preheader_length",
        "prior_email_count",
        "prior_open_count",
        "prior_click_count",
        "historical_open_rate",
        "historical_click_rate"
    ],

    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket"
    ],

    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_missing",
        "subject_contains_number",
        "subject_contains_promotion",
        "subject_contains_exclamation",
        "preheader_missing",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "preheader_contains_exclamation",
        "had_prior_click",
        "is_first_email"
    ]
}

In [106]:
# validate
feature_type_sets = {
    "A_core": feature_types_A,
    "B_interactions": feature_types_B,
    "C_expanded": feature_types_C
}

for name, type_dict in feature_type_sets.items():
    typed_features = (
        type_dict["numeric"]
        + type_dict["categorical"]
        + type_dict["binary"]
    )

    original_features = feature_sets[name]

    missing_from_types = sorted(set(original_features) - set(typed_features))
    extra_in_types = sorted(set(typed_features) - set(original_features))
    duplicate_features = pd.Series(typed_features).value_counts()
    duplicate_features = duplicate_features[duplicate_features > 1].index.tolist()

    print(name)
    print("Original feature count:", len(original_features))
    print("Typed feature count:", len(typed_features))
    print("Missing from type definitions:", missing_from_types)
    print("Extra in type definitions:", extra_in_types)
    print("Duplicated in type definitions:", duplicate_features)
    print()

A_core
Original feature count: 15
Typed feature count: 15
Missing from type definitions: []
Extra in type definitions: []
Duplicated in type definitions: []

B_interactions
Original feature count: 17
Typed feature count: 17
Missing from type definitions: []
Extra in type definitions: []
Duplicated in type definitions: []

C_expanded
Original feature count: 28
Typed feature count: 28
Missing from type definitions: []
Extra in type definitions: []
Duplicated in type definitions: []



## 2.14 Define preprocessing rules

This section defines the preprocessing rules that will later be used inside modelling pipelines. The rules are defined here, but preprocessing is not fitted globally. During modelling, preprocessing must be fitted only on the training fold and then applied to the corresponding validation fold or final test set.

In [107]:
preprocessing_rules = {
    "numeric": {
        "missing_values": "impute with 0",
        "scaling": "standardize using training-fold mean and standard deviation",
        "reason": (
            "Numerical variables are scaled for logistic regression. "
            "Historical rates are imputed with 0 while retaining is_first_email, "
            "so first-email observations remain identifiable."
        )
    },

    "categorical": {
        "missing_values": "impute with 'Missing'",
        "encoding": "one-hot encode",
        "unknown_categories": "ignore unseen categories in validation/test",
        "reason": (
            "Categorical missingness is treated as informative. "
            "One-hot encoding is required for logistic regression."
        )
    },

    "binary": {
        "missing_values": "impute with most frequent value if needed",
        "encoding": "pass through as 0/1",
        "reason": (
            "Binary indicators already contain model-ready information and do not need scaling."
        )
    }
}

preprocessing_rules

{'numeric': {'missing_values': 'impute with 0',
  'scaling': 'standardize using training-fold mean and standard deviation',
  'reason': 'Numerical variables are scaled for logistic regression. Historical rates are imputed with 0 while retaining is_first_email, so first-email observations remain identifiable.'},
 'categorical': {'missing_values': "impute with 'Missing'",
  'encoding': 'one-hot encode',
  'unknown_categories': 'ignore unseen categories in validation/test',
  'reason': 'Categorical missingness is treated as informative. One-hot encoding is required for logistic regression.'},
 'binary': {'missing_values': 'impute with most frequent value if needed',
  'encoding': 'pass through as 0/1',
  'reason': 'Binary indicators already contain model-ready information and do not need scaling.'}}

In [108]:
for name, type_dict in feature_type_sets.items():
    print(name)

    for feature_type, cols in type_dict.items():
        missing_summary = (
            trainval_df[cols]
            .isna()
            .mean()
            .sort_values(ascending=False)
        )

        print(f"\n{feature_type} missingness:")
        display(missing_summary)

    print("\n" + "-" * 60 + "\n")

A_core

numeric missingness:


n_interests    0.0
dtype: float64


categorical missingness:


age_group                       0.743214
historical_open_bucket          0.021908
gender                          0.000000
campaign_topic                  0.000000
subject_length_group            0.000000
preheader_length_group          0.000000
prior_email_frequency_bucket    0.000000
dtype: float64


binary missingness:


...


categorical missingness:


age_group                       0.743214
historical_open_bucket          0.021908
gender                          0.000000
campaign_topic                  0.000000
subject_length_group            0.000000
preheader_length_group          0.000000
prior_email_frequency_bucket    0.000000
dtype: float64


binary missingness:


interest_topic_match              0.0
has_campaign_metadata             0.0
subject_missing                   0.0
subject_contains_number           0.0
subject_contains_promotion        0.0
subject_contains_exclamation      0.0
preheader_missing                 0.0
preheader_contains_number         0.0
preheader_contains_promotion      0.0
preheader_contains_exclamation    0.0
had_prior_click                   0.0
is_first_email                    0.0
dtype: float64


------------------------------------------------------------



In [109]:
pd.crosstab(
    trainval_df["is_first_email"],
    trainval_df["historical_open_rate"].isna(),
    margins=True
)

historical_open_rate,False,True,All
is_first_email,,,
False,764698,0,764698
True,0,17128,17128
All,764698,17128,781826


## 2.15 Build preprocessing pipeline definitions

This section defines reusable preprocessing pipelines for each feature set. These preprocessing objects are not fitted globally in this notebook. During modelling, they must be fitted only on the training fold inside cross-validation.

In [110]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [111]:
def build_preprocessor(feature_types):
    numeric_features = feature_types["numeric"]
    categorical_features = feature_types["categorical"]
    binary_features = feature_types["binary"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    binary_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
            ("binary", binary_transformer, binary_features)
        ],
        remainder="drop"
    )

    return preprocessor

In [112]:
preprocessors = {
    "A_core": build_preprocessor(feature_types_A),
    "B_interactions": build_preprocessor(feature_types_B),
    "C_expanded": build_preprocessor(feature_types_C)
}

for name, preprocessor in preprocessors.items():
    print(name)
    print(preprocessor)
    print()

A_core
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=0,
                                                                strategy='constant')),
                                                 ('scaler', StandardScaler())]),
                                 ['n_interests']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['gender', 'age_group', 'campaign_t...
                                  'subject_length_group',
                          

## 2.16 Save preprocessing metadata

This section stores the final preprocessing decisions for reproducibility.

No preprocessing is fitted or applied here.

Only configuration objects and modelling settings are saved.

In [113]:
# save feature sets
preprocessing_config = {
    "target": TARGET_COL,
    "group_column": GROUP_COL,
    "final_test_size": FINAL_TEST_SIZE,
    "cv_folds": INNER_CV_SPLITS,

    "feature_sets": {
        "A_core": feature_set_A_core,
        "B_interactions": feature_set_B_interactions,
        "C_expanded": feature_set_C_expanded
    },

    "feature_types": {
        "A_core": feature_types_A,
        "B_interactions": feature_types_B,
        "C_expanded": feature_types_C
    }
}

In [114]:
# validation
print("Target:", preprocessing_config["target"])
print("Group:", preprocessing_config["group_column"])
print()

for name, features in preprocessing_config["feature_sets"].items():
    print(name)
    print("Features:", len(features))
    print()

Target: open
Group: mailing_id

A_core
Features: 15

B_interactions
Features: 17

C_expanded
Features: 28



In [115]:
# save to disk
import json

CONFIG_PATH = OUTPUT_DIR / "open_preprocessing_config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(preprocessing_config, f, indent=4)

print(CONFIG_PATH)

/Users/khanhhien/Documents/Thesis/Outputs/open_preprocessing_config.json
